# PDF parsing experiments

In [1]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from ingestion.pdf_parser import PDFParser, ParsedDocument

In [2]:
# Point at any local PDF to exercise the parser
PDF_PATH = Path("../data/pdfs/example.pdf")

parser = PDFParser(extract_images=True)
# doc: ParsedDocument = parser.parse(PDF_PATH)
# print(f"Pages: {doc.total_pages}")
# print(doc.pages[0].markdown[:500])

NameError: name 'PDFParser' is not defined

In [1]:
import fitz
import pymupdf4llm
from pathlib import Path

pdf_path = Path("../data/pdfs/attention_is_all_you_need.pdf")
doc = fitz.open(str(pdf_path))

# Inspect pages 0, 3, 6 — title, content, results
for page_num in [0, 3, 6]:
    page = doc[page_num]
    markdown = pymupdf4llm.to_markdown(str(pdf_path), pages=[page_num])
    
    word_count = len(markdown.split())
    has_table = bool(__import__('re').search(r'\|[\s\-:]+\|', markdown))
    raster = page.get_images(full=True)
    drawings = page.get_drawings()
    
    print(f"\n{'='*50}")
    print(f"PAGE {page_num}")
    print(f"{'='*50}")
    print(f"Word count:     {word_count}")
    print(f"Has table:      {has_table}")
    print(f"Raster images:  {len(raster)}")
    print(f"Vector drawings:{len(drawings)}")
    print(f"\n--- Markdown preview (first 500 chars) ---")
    print(markdown[:500])


PAGE 0
Word count:     356
Has table:      False
Raster images:  0
Vector drawings:3

--- Markdown preview (first 500 chars) ---
# **Language Models are Few-Shot Learners** 

**Tom B. Brown** _[∗]_ 

**Benjamin Mann** _[∗]_ **Nick Ryder** _[∗]_ **Melanie Subbiah** _[∗]_ 

**Jared Kaplan** _[†]_ **Prafulla Dhariwal Arvind Neelakantan Pranav Shyam Girish Sastry Amanda Askell Sandhini Agarwal Ariel Herbert-Voss Gretchen Krueger Tom Henighan** 

**Rewon Child Aditya Ramesh** 

**Daniel M. Ziegler Jeffrey Wu Clemens Winter** 

**Christopher Hesse Mark Chen** 

**Eric Sigler Mateusz Litwin Scott Gray** 

**Benjamin Chess** 

**

PAGE 3
Word count:     619
Has table:      False
Raster images:  1
Vector drawings:1

--- Markdown preview (first 500 chars) ---
**==> picture [376 x 209] intentionally omitted <==**

**Figure 1.2: Larger models make increasingly efficient use of in-context information.** We show in-context learning performance on a simple task requiring the model to remove random s

In [2]:
pdf_path = Path("../data/pdfs/vit_image_worth_16x16.pdf")
doc = fitz.open(str(pdf_path))

for page_num in [0, 2, 4, 7]:
    page = doc[page_num]
    markdown = pymupdf4llm.to_markdown(str(pdf_path), pages=[page_num])
    
    import re
    word_count = len(markdown.split())
    has_table = bool(re.search(r'\|[\s\-:]+\|', markdown))
    raster = page.get_images(full=True)
    drawings = page.get_drawings()
    omitted = markdown.count("intentionally omitted")
    eq_markers = len(re.findall(r'\$+|\\\[|\\\(', markdown))
    
    print(f"\n{'='*50}")
    print(f"PAGE {page_num}")
    print(f"{'='*50}")
    print(f"Word count:       {word_count}")
    print(f"Has table:        {has_table}")
    print(f"Raster images:    {len(raster)}")
    print(f"Vector drawings:  {len(drawings)}")
    print(f"Omitted markers:  {omitted}")
    print(f"Equation markers: {eq_markers}")
    print(f"\n--- Markdown preview (first 400 chars) ---")
    print(markdown[:400])


PAGE 0
Word count:       508
Has table:        False
Raster images:    0
Vector drawings:  2
Omitted markers:  0
Equation markers: 0

--- Markdown preview (first 400 chars) ---
Published as a conference paper at ICLR 2021 

# AN IMAGE IS WORTH 16X16 WORDS: TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE 

**Alexey Dosovitskiy** _[∗][,][†]_ **, Lucas Beyer** _[∗]_ **, Alexander Kolesnikov** _[∗]_ **, Dirk Weissenborn** _[∗]_ **, Xiaohua Zhai** _[∗]_ **, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer, Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby** _[∗][

PAGE 2
Word count:       444
Has table:        False
Raster images:    2
Vector drawings:  333
Omitted markers:  1
Equation markers: 0

--- Markdown preview (first 400 chars) ---
Published as a conference paper at ICLR 2021 

**==> picture [330 x 167] intentionally omitted <==**

Figure 1: Model overview. We split an image into fixed-size patches, linearly embed each of them, add position embeddings, and feed the res

In [3]:
pdf_path = Path("../data/pdfs/vit_image_worth_16x16.pdf")
doc = fitz.open(str(pdf_path))
import re

def detect_content_types(page_markdown, page):
    table_pattern = re.compile(r'\|[\s\-:]+\|')
    has_tables = bool(table_pattern.search(page_markdown))
    table_count = 1 if has_tables else 0

    equation_markers = re.findall(r'\$+|\\\[|\\\(', page_markdown)
    equation_count = len(equation_markers)
    has_equations = equation_count > 3

    raster_images = page.get_images(full=True)
    vector_drawings = page.get_drawings()
    figure_count = len(raster_images)
    drawing_count = len(vector_drawings)
    omitted_count = page_markdown.count("intentionally omitted")

    has_figures = (
        figure_count > 0
        or omitted_count > 0
        or drawing_count > 50
    )

    counts = {
        "figure_count": figure_count,
        "drawing_count": drawing_count,
        "table_count": table_count,
        "equation_count": equation_count,
        "omitted_count": omitted_count,
    }
    return has_figures, has_tables, has_equations, counts

# Expected results:
# Page 0 → figures=F, tables=F, equations=F (title page)
# Page 2 → figures=T, tables=F, equations=F (figure page, 333 drawings + omitted)
# Page 4 → figures=F, tables=T, equations=F (table page)
# Page 7 → figures=T, tables=F, equations=F (6 raster images)

for page_num, expected in [(0, "title"), (2, "figure"), (4, "table"), (7, "figure")]:
    page = doc[page_num]
    md = pymupdf4llm.to_markdown(str(pdf_path), pages=[page_num])
    hf, ht, he, counts = detect_content_types(md, page)
    print(f"Page {page_num} ({expected:8s}) → figures={hf}, tables={ht}, equations={he} | {counts}")

Page 0 (title   ) → figures=False, tables=False, equations=False | {'figure_count': 0, 'drawing_count': 2, 'table_count': 0, 'equation_count': 0, 'omitted_count': 0}
Page 2 (figure  ) → figures=True, tables=False, equations=False | {'figure_count': 2, 'drawing_count': 333, 'table_count': 0, 'equation_count': 0, 'omitted_count': 1}
Page 4 (table   ) → figures=False, tables=True, equations=False | {'figure_count': 0, 'drawing_count': 4, 'table_count': 1, 'equation_count': 0, 'omitted_count': 0}
Page 7 (figure  ) → figures=True, tables=False, equations=False | {'figure_count': 6, 'drawing_count': 8, 'table_count': 0, 'equation_count': 0, 'omitted_count': 1}


In [2]:
from ingestion.pdf_parser import PDFParser
from pathlib import Path

parser = PDFParser(image_output_dir=Path("../data/images"), extract_images=True)
doc = parser.parse(Path("../data/pdfs/vit_image_worth_16x16.pdf"))

print(f"Title:       {doc.title}")
print(f"Total pages: {doc.total_pages}")
print(f"\nPer-page summary:")
for page in doc.pages:
    route = []
    if page.has_figures: route.append("fig")
    if page.has_tables:  route.append("tbl")
    if page.has_equations: route.append("eq")
    flags = ",".join(route) if route else "text"
    print(f"  p{page.page_number:02d} | {page.word_count:4d} words | {flags:12s} | images saved: {len(page.image_paths)}")

Title:       vit_image_worth_16x16
Total pages: 22

Per-page summary:
  p00 |  508 words | text         | images saved: 0
  p01 |  727 words | text         | images saved: 0
  p02 |  444 words | fig          | images saved: 2
  p03 |  599 words | fig          | images saved: 0
  p04 |  733 words | tbl          | images saved: 0
  p05 |  485 words | fig,tbl      | images saved: 0
  p06 |  451 words | fig          | images saved: 0
  p07 |  640 words | fig          | images saved: 6
  p08 |  519 words | fig          | images saved: 30
  p09 |  484 words | text         | images saved: 0
  p10 |  492 words | text         | images saved: 0
  p11 |  369 words | text         | images saved: 0
  p12 |  385 words | fig,tbl      | images saved: 0
  p13 |  627 words | tbl          | images saved: 0
  p14 |  217 words | tbl          | images saved: 0
  p15 |  459 words | fig,tbl      | images saved: 0
  p16 |  412 words | fig,tbl      | images saved: 0
  p17 |  493 words | fig          | images sa

In [4]:
import pymupdf4llm
chunks = pymupdf4llm.to_markdown("../data/pdfs/vit_image_worth_16x16.pdf", page_chunks=True)
print(chunks[0]["metadata"].keys())
print(chunks[0]["metadata"])

dict_keys(['format', 'title', 'author', 'subject', 'keywords', 'creator', 'producer', 'creationDate', 'modDate', 'trapped', 'encryption', 'file_path', 'page_count', 'page_number'])
{'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.21', 'creationDate': 'D:20210604001958Z', 'modDate': 'D:20210604001958Z', 'trapped': '', 'encryption': None, 'file_path': '../data/pdfs/vit_image_worth_16x16.pdf', 'page_count': 22, 'page_number': 1}


# Page router signal testing

In [ ]:
from ingestion.page_router import PageRouter, RouterConfig, ProcessingRoute

router = PageRouter(RouterConfig(
    figure_threshold=1,
    table_threshold=1,
    equation_density_threshold=0.15,
))

In [3]:
from ingestion.page_router import PageRouter, RouterConfig

router = PageRouter()
decisions = router.route_document(doc.pages)
text_pages, vlm_pages = router.split_by_route(doc.pages, decisions)

print(f"TEXT_ONLY:    {sum(1 for d in decisions if d.route.value == 'text_only')}")
print(f"VLM_REQUIRED: {sum(1 for d in decisions if d.route.value == 'vlm')}")
print(f"SKIP:         {sum(1 for d in decisions if d.route.value == 'skip')}")

print(f"\nPer-page routing:")
for d in decisions:
    print(f"  p{d.page_number:02d} | {d.route.value:12s} | conf={d.confidence:.2f} | {d.reasons[0]}")

TEXT_ONLY:    4
VLM_REQUIRED: 16
SKIP:         2

Per-page routing:
  p00 | text_only    | conf=0.85 | no visual complexity detected
  p01 | vlm          | conf=0.65 | no visual complexity detected
  p02 | vlm          | conf=0.75 | figures detected (raster=2, omitted=1, drawings=333)
  p03 | vlm          | conf=0.75 | figures detected (raster=0, omitted=1, drawings=1)
  p04 | vlm          | conf=0.75 | table detected
  p05 | vlm          | conf=0.90 | figures detected (raster=0, omitted=1, drawings=63)
  p06 | vlm          | conf=0.75 | figures detected (raster=0, omitted=3, drawings=378)
  p07 | vlm          | conf=0.75 | figures detected (raster=6, omitted=1, drawings=8)
  p08 | skip         | conf=0.90 | structural page: references/ToC/acknowledgments or too short
  p09 | text_only    | conf=0.85 | no visual complexity detected
  p10 | text_only    | conf=0.85 | no visual complexity detected
  p11 | text_only    | conf=0.85 | no visual complexity detected
  p12 | skip         | con

In [4]:
for page in doc.pages:
    if page.page_number in [8, 12]:
        print(f"\n--- PAGE {page.page_number} (first 300 chars) ---")
        print(page.markdown[:300])


--- PAGE 8 (first 300 chars) ---
Published as a conference paper at ICLR 2021 

**==> picture [396 x 120] intentionally omitted <==**

**----- Start of picture text -----**<br>
RGB embedding filters<br>(first 28 principal components) Position embedding similarity ViT-L/16<br>1<br>1 120<br>2 100<br>3 80<br>4 60<br>5 40 Head 1Head 2<

--- PAGE 12 (first 300 chars) ---
Published as a conference paper at ICLR 2021 

|Models|Dataset|Epochs|Base LR|LR decay|Weight decay|Dropout|
|---|---|---|---|---|---|---|
|ViT-B/_{_16,32_}_|JFT-300M|7|8_·_10_−_4|linear|0.1|0.0|
|ViT-L/32|JFT-300M|7|6_·_10_−_4|linear|0.1|0.0|
|ViT-L/16|JFT-300M|7/14|4_·_10_−_4|linear|0.1|0.0|
|ViT-


In [5]:
router2 = PageRouter()
for page in doc.pages:
    if page.page_number in [8, 12]:
        print(f"\n--- PAGE {page.page_number} ---")
        print(f"word_count: {page.word_count}")
        print(f"min_word_count config: {router2.config.min_word_count}")
        
        import re
        heading_pattern = re.compile(
            r'^#{1,3}\s+(references|bibliography|acknowledgment|acknowledgement|table of contents|appendix|author contributions|funding|declaration)',
            re.MULTILINE | re.IGNORECASE
        )
        match = heading_pattern.search(page.markdown)
        print(f"heading match: {match}")
        print(f"markdown first 200: {page.markdown[:200]}")


--- PAGE 8 ---
word_count: 519
min_word_count config: 10
heading match: <re.Match object; span=(2829, 2847), match='## ACKNOWLEDGEMENT'>
markdown first 200: Published as a conference paper at ICLR 2021 

**==> picture [396 x 120] intentionally omitted <==**

**----- Start of picture text -----**<br>
RGB embedding filters<br>(first 28 principal components)

--- PAGE 12 ---
word_count: 385
min_word_count config: 10
heading match: <re.Match object; span=(1085, 1096), match='## APPENDIX'>
markdown first 200: Published as a conference paper at ICLR 2021 

|Models|Dataset|Epochs|Base LR|LR decay|Weight decay|Dropout|
|---|---|---|---|---|---|---|
|ViT-B/_{_16,32_}_|JFT-300M|7|8_·_10_−_4|linear|0.1|0.0|
|ViT


In [6]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser

# First run downloads model weights — will take a minute
config_parser = ConfigParser({"output_format": "json"})
converter = PdfConverter(
    config=config_parser.generate_config_dict(),
    artifact_dict=create_model_dict(),
    processor_list=config_parser.get_processors(),
    renderer=config_parser.get_renderer(),
)

from pathlib import Path
rendered = converter(str(Path("../data/pdfs/vit_image_worth_16x16.pdf")))

# Understand what we got back
print(type(rendered))
print(dir(rendered))



















2026-04-27 16:36:49,020 [WARNING] surya: `TableRecEncoderDecoderModel` is not compatible with mps backend. Defaulting to cpu instead




























































































Recognizing Layout:   0%|          | 0/22 [00:00<?, ?it/s]/Users/monish/Desktop/Monish-Personal/paper-intelligence/.venv/lib/python3.11/site-packages/surya/common/surya/encoder/__init__.py:491: UserWarning: MPS: nonzero op is not natively supported for the provided input on MacOS14Falling back on CPU. This may have performance implications.See github.com/pytorch/pytorch/issues/122916 for further info (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/mps/operations/Indexing.mm:378.)
  attention_mask[valid_2d] = 0


RuntimeError: MPS backend out of memory (MPS allocated: 6.15 GiB, other allocations: 2.50 GiB, max allowed: 9.07 GiB). Tried to allocate 634.57 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [2]:
import asyncio
from storage.vector_store import QdrantVectorStore, VectorStoreConfig, VectorDocument

async def test():
    config = VectorStoreConfig(
        embedding_dim=384,
        collection_name="paper_chunks_test",
    )
    store = QdrantVectorStore(config)

    # Test upsert
    docs = [
        VectorDocument(
            chunk_id="abc123",
            text="Vision Transformer applies self-attention to image patches",
            embedding=[0.1] * 384,
            metadata={"section_type": "methodology", "chunk_type": "text"},
        ),
        VectorDocument(
            chunk_id="def456",
            text="We evaluate on ImageNet and CIFAR-10 benchmarks",
            embedding=[0.2] * 384,
            metadata={"section_type": "experiments", "chunk_type": "text"},
        ),
    ]
    n = await store.upsert(docs)
    print(f"Upserted: {n}")

    # Test count
    total = await store.count()
    print(f"Total vectors: {total}")

    # Test search
    results = await store.search(query_embedding=[0.1] * 384, top_k=2)
    for r in results:
        print(f"score={r.score:.3f} | {r.text[:60]}")

    # Test filtered search
    results = await store.search(
        query_embedding=[0.1] * 384,
        top_k=2,
        filters={"section_type": "methodology"},
    )
    print(f"Filtered results: {len(results)}")

    # Test delete
    deleted = await store.delete(["abc123", "def456"])
    print(f"Deleted: {deleted}")

    total = await store.count()
    print(f"Total after delete: {total}")

await test()

Upserted: 2
Total vectors: 2
score=1.000 | Vision Transformer applies self-attention to image patches
score=1.000 | We evaluate on ImageNet and CIFAR-10 benchmarks
Filtered results: 1
Deleted: 2
Total after delete: 0


In [8]:
import inspect
from storage.vector_store import QdrantVectorStore
print(inspect.getmembers(QdrantVectorStore, predicate=inspect.isfunction))

[('__init__', <function QdrantVectorStore.__init__ at 0x11a3f0680>), ('_get_client', <function QdrantVectorStore._get_client at 0x11a3f0720>), ('_to_uuid', <function QdrantVectorStore._to_uuid at 0x11a3f07c0>), ('count', <function VectorStoreBase.count at 0x11a3f05e0>), ('delete', <function VectorStoreBase.delete at 0x11a3f0540>), ('search', <function QdrantVectorStore.search at 0x11a3f0900>), ('upsert', <function QdrantVectorStore.upsert at 0x11a3f0860>)]


In [2]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [3]:
from dotenv import load_dotenv
load_dotenv("../.env")

from extraction.embedder import Embedder, EmbedderConfig

embedder = Embedder()

# Test single embedding
vector = await embedder.embed_one("Vision Transformer applies self-attention to image patches")
print(f"Dimension: {len(vector)}")
print(f"First 5 values: {vector[:5]}")

# Test batch embedding
texts = [
    "We evaluate on ImageNet and achieve 88.5% top-1 accuracy",
    "Our method outperforms ResNet by a significant margin",
    "The attention mechanism allows the model to focus on relevant patches",
]
vectors = await embedder.embed(texts)
print(f"Batch size: {len(vectors)}")
print(f"Each vector dimension: {len(vectors[0])}")

# Quick sanity check — similar texts should have higher cosine similarity
import numpy as np
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_01 = cosine_sim(vectors[0], vectors[1])
sim_02 = cosine_sim(vectors[0], vectors[2])
print(f"Similarity (accuracy vs outperforms): {sim_01:.3f}")
print(f"Similarity (accuracy vs attention):   {sim_02:.3f}")

Dimension: 1536
First 5 values: [-0.01496124267578125, 0.01351165771484375, -0.031524658203125, -0.0029659271240234375, -0.0241851806640625]
Batch size: 3
Each vector dimension: 1536
Similarity (accuracy vs outperforms): 0.558
Similarity (accuracy vs attention):   0.324


In [2]:
from extraction.vlm_client import VLMClient, SGLangConfig, ExtractionRequest
from extraction.schemas import PartialExtraction, Author, Dataset, Method, FieldConfidence

client = VLMClient()

# Test _build_extraction_prompt
prompt_all = client._build_extraction_prompt(None)
prompt_filtered = client._build_extraction_prompt(["title", "authors", "abstract"])
print("=== Full prompt (first 300 chars) ===")
print(prompt_all[:300])
print("\n=== Filtered prompt ===")
print(prompt_filtered)

# Test _build_messages — text only
request = ExtractionRequest(
    page_text="Vision Transformer applies self-attention to image patches...",
    user_prompt=prompt_filtered,
)
messages = client._build_messages(request)
print(f"\n=== Messages structure ===")
print(f"Num messages: {len(messages)}")
print(f"Roles: {[m['role'] for m in messages]}")
print(f"User content blocks: {len(messages[1]['content'])}")
print(f"Block types: {[b['type'] for b in messages[1]['content']]}")

# Test merge_extractions
p1 = PartialExtraction(
    title="An Image is Worth 16x16 Words",
    problem_statement="",
    authors=[Author(name="Dosovitskiy", affiliations=["Google Brain"])],
    datasets=[Dataset(name="ImageNet")],
    extraction_confidence=0.9,
    field_confidence=FieldConfidence(title=0.95, authors=0.9, datasets=0.0),
)
p2 = PartialExtraction(
    title="",
    problem_statement="CNNs dominate vision but transformers could work better",
    datasets=[Dataset(name="CIFAR-10"), Dataset(name="ImageNet")],
    methods=[Method(name="ViT", description="Vision Transformer")],
    extraction_confidence=0.7,
    field_confidence=FieldConfidence(
        title=0.0, problem_statement=0.85, datasets=0.8, methods=0.9
    ),
)

merged = client.merge_extractions([p1, p2])
print(f"\n=== Merged extraction ===")
print(f"title:              {merged.title}")
print(f"problem_statement:  {merged.problem_statement}")
print(f"authors:            {[a.name for a in merged.authors]}")
print(f"datasets:           {[d.name for d in merged.datasets]}")
print(f"methods:            {[m.name for m in merged.methods]}")
print(f"confidence:         {merged.extraction_confidence}")
print(f"field_confidence:   {merged.field_confidence}")
print(f"metadata:           {merged.extraction_metadata}")

=== Full prompt (first 300 chars) ===
Extract the following fields from this research paper page:

- title: Full paper title exactly as written
- authors: All authors with names, affiliations, emails if present
- institutions: All affiliated institutions with name, department, country
- problem_statement: 1-3 sentences: what problem doe

=== Filtered prompt ===
Extract the following fields from this research paper page:

- title: Full paper title exactly as written
- authors: All authors with names, affiliations, emails if present
- abstract: Full abstract text verbatim

Rules:
- Only extract information explicitly present in the provided content
- Do not infer, hallucinate, or fill gaps with prior knowledge
- For fields not present on this page, use empty string or empty list
- Extract confidence scores per field: 1.0 if clearly present, 0.5 if partially present, 0.0 if absent
- Return valid JSON matching the PartialExtraction schema exactly

=== Messages structure ===
Num messages: 2

In [ ]:
# Uncomment once PDFParser.parse() is implemented
# decisions = router.route_document(doc.pages)
# for d in decisions:
#     print(f"p{d.page_number:>3}  {d.route.value:<14}  conf={d.confidence:.2f}  {d.reasons}")

In [ ]:
# Route distribution
# from collections import Counter
# Counter(d.route for d in decisions)

# Chunker output inspection

In [ ]:
from ingestion.chunker import SectionAwareChunker, ChunkerConfig

chunker = SectionAwareChunker(ChunkerConfig(
    max_tokens=512,
    overlap_tokens=64,
))

In [ ]:
# Uncomment once router is implemented
# text_pages, vlm_pages = router.split_by_route(doc.pages, decisions)
# chunks = chunker.chunk_document(text_pages, source_path=str(PDF_PATH))

# print(f"Total chunks: {len(chunks)}")
# print(f"Sections found: {sorted({c.section_type.value for c in chunks})}")

In [ ]:
# Inspect a sample chunk
# sample = chunks[0]
# print(f"ID:       {sample.chunk_id}")
# print(f"Section:  {sample.section_type.value}")
# print(f"Heading:  {sample.section_heading}")
# print(f"Pages:    {sample.page_numbers}")
# print(f"Tokens:   {sample.token_estimate}")
# print()
# print(sample.text[:400])

In [ ]:
# Token distribution across chunks
# import statistics
# token_counts = [c.token_estimate for c in chunks]
# print(f"min={min(token_counts)}  max={max(token_counts)}  mean={statistics.mean(token_counts):.0f}  median={statistics.median(token_counts):.0f}")